# Experiment 02 — Engineered-Feature Random Forest

## Purpose and decision rule

The 20 Hz representation preserved the original fused Random Forest baseline,
achieving mean subject-fold macro-F1 **0.6882 ± 0.0509**.

This experiment asks whether deterministic temporal and frequency-domain
features capture useful activity structure beyond the original whole-window
statistics.

All comparisons use:

- the same 25,585 paired development examples;
- the same five subject-wise folds;
- the same 300-tree Random Forest;
- the same evaluation metrics;
- no hyperparameter tuning;
- no test data.

Four predefined candidates will be compared with the 56-feature baseline:

1. baseline plus distribution-shape and cross-axis features;
2. baseline plus temporal-dynamics features;
3. baseline plus frequency-domain features;
4. all engineered feature groups combined.

A feature expansion is considered practically useful if it improves mean
macro-F1 by at least **0.01** over 0.6882 and improves at least three of the five
subject folds. This is a practical complexity gate rather than a formal
significance test.

Feature groups and formulas will be fixed before their model results are
examined.

## 1. Fixed engineered-feature specification

All engineered features are deterministic functions of a single resampled
window. They do not use activity labels or population-level fitted parameters.

Features are calculated independently for accelerometer and gyroscope. Within
each sensor, `x`, `y`, `z`, and magnitude are treated as four signals.

### 1.1 Baseline features

The established parity representation contains seven statistics per signal:

- mean;
- sample standard deviation;
- minimum;
- maximum;
- median;
- interquartile range;
- RMS.

This produces 28 features per sensor and 56 fused features.

### 1.2 Distribution shape and cross-axis relationships

For each of the four signals:

- 10th, 25th, 75th, and 90th percentiles;
- standardized third central moment (skewness);
- excess standardized fourth central moment (kurtosis).

For the three physical axes:

- Pearson correlation between `x` and `y`;
- Pearson correlation between `x` and `z`;
- Pearson correlation between `y` and `z`.

Constant-signal correlations and standardized moments are defined as zero.

This group adds 27 features per sensor and 54 fused features.

### 1.3 Temporal dynamics

The first-difference rate is calculated using the fixed 0.05-second timestep.
For each of the four signals, we calculate:

- mean absolute rate;
- sample standard deviation of the rate;
- RMS rate;
- maximum absolute rate.

We also calculate the magnitude of the three-axis rate vector and summarize it
using mean, sample standard deviation, RMS, and maximum.

This group adds 20 features per sensor and 40 fused features.

### 1.4 Frequency-domain structure

A Hann-window periodogram is calculated after removing each signal's mean. The
frequency grid spans 0–10 Hz with 0.2 Hz resolution. DC is excluded from the
frequency features.

For each of the four signals, we calculate:

- total non-DC spectral power;
- dominant non-DC frequency;
- dominant-bin power as a proportion of total power;
- spectral centroid;
- normalized spectral entropy;
- relative power in `[0.2, 1.0)` Hz;
- relative power in `[1.0, 3.0)` Hz;
- relative power in `[3.0, 5.0)` Hz;
- relative power in `[5.0, 10.0]` Hz.

Dominant frequency is interpreted together with peak-power ratio and spectral
entropy, so weak or diffuse spectra are not treated as strongly periodic.

This group adds 36 features per sensor and 72 fused features.

### 1.5 Predefined model comparisons

| Model | Feature count |
|---|---:|
| Baseline | 56 |
| Baseline + distribution/correlation | 110 |
| Baseline + temporal dynamics | 96 |
| Baseline + frequency structure | 128 |
| All engineered groups | 222 |

These definitions and model combinations are fixed before any performance
results are examined.

## 2. Load the development artifact and construct the first feature groups

Only the paired-watch development artifact is loaded. The baseline features are
recreated using the same implementation validated in the parity experiment.

The distribution/correlation group is then constructed according to the fixed
specification. No labels or fold-level population statistics are used during
feature extraction.

In [1]:
import json
from pathlib import Path

import numpy as np
import pandas as pd


CURRENT_DIR = Path.cwd().resolve()

PROJECT_ROOT = next(
    (
        candidate
        for candidate in [CURRENT_DIR, *CURRENT_DIR.parents]
        if (
            candidate
            / "modified"
            / "watch_pair_20hz"
            / "preprocessing_meta.json"
        ).exists()
    ),
    None,
)

if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Could not locate the activity_project root."
    )

ARTIFACT_DIR = (
    PROJECT_ROOT
    / "modified"
    / "watch_pair_20hz"
)

TRAIN_SEQUENCE_PATH = (
    ARTIFACT_DIR / "train_sequences.npz"
)
TRAIN_METADATA_PATH = (
    ARTIFACT_DIR / "train_metadata.parquet"
)
PREPROCESSING_META_PATH = (
    ARTIFACT_DIR / "preprocessing_meta.json"
)

preprocessing_meta = json.loads(
    PREPROCESSING_META_PATH.read_text(
        encoding="utf-8"
    )
)

with np.load(TRAIN_SEQUENCE_PATH) as artifact:
    train_accel = artifact["accel"]
    train_gyro = artifact["gyro"]
    target_offsets_s = artifact[
        "target_offsets_s"
    ]

train_metadata = pd.read_parquet(
    TRAIN_METADATA_PATH
)

assert train_accel.shape == (
    25_585,
    100,
    3,
)
assert train_gyro.shape == train_accel.shape
assert len(train_metadata) == len(train_accel)

y = train_metadata[
    "activity_code"
].astype(str).to_numpy()

fold_assignments = train_metadata[
    "fold"
].astype(int).to_numpy()

subject_ids = train_metadata[
    "user_id"
].to_numpy()

print(
    f"Loaded {len(train_metadata):,} "
    "development examples."
)

Loaded 25,585 development examples.


In [2]:
SIGNAL_NAMES = [
    "x",
    "y",
    "z",
    "magnitude",
]

BASELINE_STATISTICS = [
    "mean",
    "std",
    "min",
    "max",
    "median",
    "iqr",
    "rms",
]


def add_magnitude(sequences):
    xyz = np.asarray(
        sequences,
        dtype=np.float64,
    )

    magnitude = np.sqrt(
        np.sum(
            np.square(xyz),
            axis=2,
            keepdims=True,
        )
    )

    return np.concatenate(
        [xyz, magnitude],
        axis=2,
    )


def extract_baseline_features(
    signals,
    sensor_prefix,
):
    signals = signals.astype(
        np.float64,
        copy=False,
    )

    first_quartile, third_quartile = (
        np.quantile(
            signals,
            [0.25, 0.75],
            axis=1,
            method="linear",
        )
    )

    statistic_values = {
        "mean": signals.mean(axis=1),
        "std": signals.std(
            axis=1,
            ddof=1,
        ),
        "min": signals.min(axis=1),
        "max": signals.max(axis=1),
        "median": np.median(
            signals,
            axis=1,
        ),
        "iqr": (
            third_quartile
            - first_quartile
        ),
        "rms": np.sqrt(
            np.mean(
                np.square(signals),
                axis=1,
            )
        ),
    }

    feature_names = [
        (
            f"{sensor_prefix}_"
            f"{signal_name}_"
            f"{statistic_name}"
        )
        for signal_name in SIGNAL_NAMES
        for statistic_name
        in BASELINE_STATISTICS
    ]

    feature_matrix = np.column_stack([
        statistic_values[
            statistic_name
        ][:, signal_index]
        for signal_index, signal_name
        in enumerate(SIGNAL_NAMES)
        for statistic_name
        in BASELINE_STATISTICS
    ])

    return feature_matrix, feature_names


accel_signals = add_magnitude(
    train_accel
)
gyro_signals = add_magnitude(
    train_gyro
)

(
    accel_baseline_features,
    accel_baseline_names,
) = extract_baseline_features(
    accel_signals,
    "accel",
)

(
    gyro_baseline_features,
    gyro_baseline_names,
) = extract_baseline_features(
    gyro_signals,
    "gyro",
)

X_baseline = np.concatenate(
    [
        accel_baseline_features,
        gyro_baseline_features,
    ],
    axis=1,
)

baseline_feature_names = (
    accel_baseline_names
    + gyro_baseline_names
)

In [3]:
DISTRIBUTION_STATISTICS = [
    "p10",
    "p25",
    "p75",
    "p90",
    "skewness",
    "excess_kurtosis",
]

AXIS_PAIRS = [
    (0, 1, "xy"),
    (0, 2, "xz"),
    (1, 2, "yz"),
]


def extract_distribution_features(
    signals,
    sensor_prefix,
):
    signals = signals.astype(
        np.float64,
        copy=False,
    )

    (
        percentile_10,
        percentile_25,
        percentile_75,
        percentile_90,
    ) = np.quantile(
        signals,
        [0.10, 0.25, 0.75, 0.90],
        axis=1,
        method="linear",
    )

    centered = (
        signals
        - signals.mean(
            axis=1,
            keepdims=True,
        )
    )

    second_moment = np.mean(
        centered**2,
        axis=1,
    )
    third_moment = np.mean(
        centered**3,
        axis=1,
    )
    fourth_moment = np.mean(
        centered**4,
        axis=1,
    )

    nonconstant = (
        second_moment
        > np.finfo(np.float64).eps
    )

    skewness = np.zeros_like(
        second_moment
    )
    excess_kurtosis = np.zeros_like(
        second_moment
    )

    np.divide(
        third_moment,
        second_moment**1.5,
        out=skewness,
        where=nonconstant,
    )

    np.divide(
        fourth_moment,
        second_moment**2,
        out=excess_kurtosis,
        where=nonconstant,
    )

    excess_kurtosis[
        nonconstant
    ] -= 3.0

    statistic_values = {
        "p10": percentile_10,
        "p25": percentile_25,
        "p75": percentile_75,
        "p90": percentile_90,
        "skewness": skewness,
        "excess_kurtosis": (
            excess_kurtosis
        ),
    }

    distribution_names = [
        (
            f"{sensor_prefix}_"
            f"{signal_name}_"
            f"{statistic_name}"
        )
        for signal_name in SIGNAL_NAMES
        for statistic_name
        in DISTRIBUTION_STATISTICS
    ]

    distribution_matrix = np.column_stack([
        statistic_values[
            statistic_name
        ][:, signal_index]
        for signal_index, signal_name
        in enumerate(SIGNAL_NAMES)
        for statistic_name
        in DISTRIBUTION_STATISTICS
    ])

    xyz = signals[:, :, :3]

    centered_xyz = (
        xyz
        - xyz.mean(
            axis=1,
            keepdims=True,
        )
    )

    correlation_columns = []
    correlation_names = []

    for (
        first_axis,
        second_axis,
        pair_name,
    ) in AXIS_PAIRS:
        first_values = centered_xyz[
            :,
            :,
            first_axis,
        ]
        second_values = centered_xyz[
            :,
            :,
            second_axis,
        ]

        numerator = np.sum(
            first_values * second_values,
            axis=1,
        )

        denominator = np.sqrt(
            np.sum(
                first_values**2,
                axis=1,
            )
            * np.sum(
                second_values**2,
                axis=1,
            )
        )

        correlation = np.zeros(
            len(signals),
            dtype=np.float64,
        )

        np.divide(
            numerator,
            denominator,
            out=correlation,
            where=(
                denominator
                > np.finfo(
                    np.float64
                ).eps
            ),
        )

        correlation_columns.append(
            correlation
        )
        correlation_names.append(
            (
                f"{sensor_prefix}_"
                f"{pair_name}_correlation"
            )
        )

    feature_matrix = np.column_stack([
        distribution_matrix,
        *correlation_columns,
    ])

    feature_names = (
        distribution_names
        + correlation_names
    )

    return feature_matrix, feature_names

In [4]:
(
    accel_distribution_features,
    accel_distribution_names,
) = extract_distribution_features(
    accel_signals,
    "accel",
)

(
    gyro_distribution_features,
    gyro_distribution_names,
) = extract_distribution_features(
    gyro_signals,
    "gyro",
)

X_distribution = np.concatenate(
    [
        accel_distribution_features,
        gyro_distribution_features,
    ],
    axis=1,
)

distribution_feature_names = (
    accel_distribution_names
    + gyro_distribution_names
)

X_baseline_distribution = np.concatenate(
    [
        X_baseline,
        X_distribution,
    ],
    axis=1,
)

baseline_distribution_names = (
    baseline_feature_names
    + distribution_feature_names
)

assert X_baseline.shape == (
    len(train_metadata),
    56,
)
assert X_distribution.shape == (
    len(train_metadata),
    54,
)
assert X_baseline_distribution.shape == (
    len(train_metadata),
    110,
)
assert np.isfinite(
    X_baseline_distribution
).all()

feature_group_overview = pd.DataFrame([
    {
        "feature_group": "Baseline",
        "added_features": 56,
        "candidate_total": 56,
    },
    {
        "feature_group": (
            "Distribution/correlation"
        ),
        "added_features": 54,
        "candidate_total": 110,
    },
])

display(feature_group_overview)

print(
    "Baseline and distribution/correlation "
    "features constructed."
)

,feature_group,added_features,candidate_total
0,Baseline,56,56
1,Distribution/correlation,54,110


Baseline and distribution/correlation features constructed.


## 3. Construct temporal-dynamics features

Because every sequence now uses the same 0.05-second timestep, first
differences can be expressed as rates per second.

The scalar rate features describe changes along each axis and in signal
magnitude. The vector-rate magnitude separately measures the total three-axis
change, which is not equivalent to the rate of the scalar signal magnitude.

In [5]:
TARGET_RATE_HZ = float(
    preprocessing_meta[
        "resampling"
    ]["target_rate_hz"]
)
TARGET_STEP_S = 1.0 / TARGET_RATE_HZ

assert np.allclose(
    np.diff(target_offsets_s),
    TARGET_STEP_S,
    atol=1e-6,
)

TEMPORAL_STATISTICS = [
    "mean_abs",
    "std",
    "rms",
    "max_abs",
]


def extract_temporal_features(
    signals,
    sensor_prefix,
    step_s,
):
    signals = signals.astype(
        np.float64,
        copy=False,
    )

    rates = (
        np.diff(
            signals,
            axis=1,
        )
        / step_s
    )

    absolute_rates = np.abs(rates)

    statistic_values = {
        "mean_abs": absolute_rates.mean(
            axis=1
        ),
        "std": rates.std(
            axis=1,
            ddof=1,
        ),
        "rms": np.sqrt(
            np.mean(
                rates**2,
                axis=1,
            )
        ),
        "max_abs": absolute_rates.max(
            axis=1
        ),
    }

    signal_feature_names = [
        (
            f"{sensor_prefix}_"
            f"{signal_name}_rate_"
            f"{statistic_name}"
        )
        for signal_name in SIGNAL_NAMES
        for statistic_name
        in TEMPORAL_STATISTICS
    ]

    signal_feature_matrix = np.column_stack([
        statistic_values[
            statistic_name
        ][:, signal_index]
        for signal_index, signal_name
        in enumerate(SIGNAL_NAMES)
        for statistic_name
        in TEMPORAL_STATISTICS
    ])

    vector_rate_magnitude = np.sqrt(
        np.sum(
            rates[:, :, :3] ** 2,
            axis=2,
        )
    )

    vector_feature_matrix = np.column_stack([
        vector_rate_magnitude.mean(
            axis=1
        ),
        vector_rate_magnitude.std(
            axis=1,
            ddof=1,
        ),
        np.sqrt(
            np.mean(
                vector_rate_magnitude**2,
                axis=1,
            )
        ),
        vector_rate_magnitude.max(
            axis=1
        ),
    ])

    vector_feature_names = [
        (
            f"{sensor_prefix}_"
            "vector_rate_magnitude_mean"
        ),
        (
            f"{sensor_prefix}_"
            "vector_rate_magnitude_std"
        ),
        (
            f"{sensor_prefix}_"
            "vector_rate_magnitude_rms"
        ),
        (
            f"{sensor_prefix}_"
            "vector_rate_magnitude_max"
        ),
    ]

    feature_matrix = np.column_stack([
        signal_feature_matrix,
        vector_feature_matrix,
    ])

    feature_names = (
        signal_feature_names
        + vector_feature_names
    )

    return feature_matrix, feature_names

In [6]:
(
    accel_temporal_features,
    accel_temporal_names,
) = extract_temporal_features(
    accel_signals,
    "accel",
    TARGET_STEP_S,
)

(
    gyro_temporal_features,
    gyro_temporal_names,
) = extract_temporal_features(
    gyro_signals,
    "gyro",
    TARGET_STEP_S,
)

X_temporal = np.concatenate(
    [
        accel_temporal_features,
        gyro_temporal_features,
    ],
    axis=1,
)

temporal_feature_names = (
    accel_temporal_names
    + gyro_temporal_names
)

X_baseline_temporal = np.concatenate(
    [
        X_baseline,
        X_temporal,
    ],
    axis=1,
)

baseline_temporal_names = (
    baseline_feature_names
    + temporal_feature_names
)

assert X_temporal.shape == (
    len(train_metadata),
    40,
)
assert X_baseline_temporal.shape == (
    len(train_metadata),
    96,
)
assert np.isfinite(
    X_baseline_temporal
).all()

feature_group_overview = pd.DataFrame([
    {
        "feature_group": "Baseline",
        "added_features": 56,
        "candidate_total": 56,
    },
    {
        "feature_group": (
            "Distribution/correlation"
        ),
        "added_features": 54,
        "candidate_total": 110,
    },
    {
        "feature_group": (
            "Temporal dynamics"
        ),
        "added_features": 40,
        "candidate_total": 96,
    },
])

display(feature_group_overview)

print(
    "Temporal-dynamics features constructed."
)

,feature_group,added_features,candidate_total
0,Baseline,56,56
1,Distribution/correlation,54,110
2,Temporal dynamics,40,96


Temporal-dynamics features constructed.


## 4. Construct frequency-domain features

Frequency features are calculated from a Hann-window periodogram with constant
detrending. With 100 timesteps at 20 Hz, the frequency spacing is 0.2 Hz and
the Nyquist frequency is 10 Hz.

DC is excluded. Windows with effectively zero non-DC power receive zero-valued
frequency features rather than undefined ratios.

In [7]:
from scipy.signal import periodogram


FREQUENCY_STATISTICS = [
    "total_power",
    "dominant_frequency_hz",
    "dominant_power_ratio",
    "spectral_centroid_hz",
    "spectral_entropy",
    "relative_power_0p2_1hz",
    "relative_power_1_3hz",
    "relative_power_3_5hz",
    "relative_power_5_10hz",
]

FREQUENCY_BANDS = [
    (
        "relative_power_0p2_1hz",
        0.2,
        1.0,
        False,
    ),
    (
        "relative_power_1_3hz",
        1.0,
        3.0,
        False,
    ),
    (
        "relative_power_3_5hz",
        3.0,
        5.0,
        False,
    ),
    (
        "relative_power_5_10hz",
        5.0,
        10.0,
        True,
    ),
]


def extract_frequency_features(
    signals,
    sensor_prefix,
    sampling_rate_hz,
):
    signals = signals.astype(
        np.float64,
        copy=False,
    )

    frequencies, power_density = (
        periodogram(
            signals,
            fs=sampling_rate_hz,
            window="hann",
            detrend="constant",
            return_onesided=True,
            scaling="density",
            axis=1,
        )
    )

    assert np.isclose(
        frequencies[1] - frequencies[0],
        0.2,
    )
    assert np.isclose(
        frequencies[-1],
        sampling_rate_hz / 2,
    )

    positive_frequencies = (
        frequencies[1:]
    )
    positive_power = (
        power_density[:, 1:, :]
    )

    frequency_step_hz = float(
        positive_frequencies[1]
        - positive_frequencies[0]
    )

    power_sum = positive_power.sum(
        axis=1
    )

    valid_power = (
        power_sum
        > np.finfo(np.float64).eps
    )

    safe_power_sum = np.where(
        valid_power,
        power_sum,
        1.0,
    )

    total_power = (
        power_sum * frequency_step_hz
    )

    dominant_indices = np.argmax(
        positive_power,
        axis=1,
    )

    dominant_frequency = (
        positive_frequencies[
            dominant_indices
        ]
    ).astype(np.float64)

    dominant_power = np.take_along_axis(
        positive_power,
        dominant_indices[:, None, :],
        axis=1,
    ).squeeze(axis=1)

    dominant_power_ratio = (
        dominant_power
        / safe_power_sum
    )

    spectral_centroid = (
        (
            positive_power
            * positive_frequencies[
                None,
                :,
                None,
            ]
        ).sum(axis=1)
        / safe_power_sum
    )

    normalized_power = (
        positive_power
        / safe_power_sum[:, None, :]
    )

    log_normalized_power = np.zeros_like(
        normalized_power
    )

    np.log(
        normalized_power,
        out=log_normalized_power,
        where=normalized_power > 0,
    )

    spectral_entropy = (
        -(
            normalized_power
            * log_normalized_power
        ).sum(axis=1)
        / np.log(
            len(positive_frequencies)
        )
    )

    dominant_frequency[
        ~valid_power
    ] = 0.0
    dominant_power_ratio[
        ~valid_power
    ] = 0.0
    spectral_centroid[
        ~valid_power
    ] = 0.0
    spectral_entropy[
        ~valid_power
    ] = 0.0

    statistic_values = {
        "total_power": total_power,
        "dominant_frequency_hz": (
            dominant_frequency
        ),
        "dominant_power_ratio": (
            dominant_power_ratio
        ),
        "spectral_centroid_hz": (
            spectral_centroid
        ),
        "spectral_entropy": (
            spectral_entropy
        ),
    }

    for (
        feature_name,
        lower_bound,
        upper_bound,
        include_upper,
    ) in FREQUENCY_BANDS:
        if include_upper:
            band_mask = (
                (
                    positive_frequencies
                    >= lower_bound
                )
                & (
                    positive_frequencies
                    <= upper_bound
                )
            )
        else:
            band_mask = (
                (
                    positive_frequencies
                    >= lower_bound
                )
                & (
                    positive_frequencies
                    < upper_bound
                )
            )

        relative_band_power = (
            positive_power[
                :,
                band_mask,
                :,
            ].sum(axis=1)
            / safe_power_sum
        )

        relative_band_power[
            ~valid_power
        ] = 0.0

        statistic_values[
            feature_name
        ] = relative_band_power

    feature_names = [
        (
            f"{sensor_prefix}_"
            f"{signal_name}_"
            f"{statistic_name}"
        )
        for signal_name in SIGNAL_NAMES
        for statistic_name
        in FREQUENCY_STATISTICS
    ]

    feature_matrix = np.column_stack([
        statistic_values[
            statistic_name
        ][:, signal_index]
        for signal_index, signal_name
        in enumerate(SIGNAL_NAMES)
        for statistic_name
        in FREQUENCY_STATISTICS
    ])

    return feature_matrix, feature_names

In [8]:
(
    accel_frequency_features,
    accel_frequency_names,
) = extract_frequency_features(
    accel_signals,
    "accel",
    TARGET_RATE_HZ,
)

(
    gyro_frequency_features,
    gyro_frequency_names,
) = extract_frequency_features(
    gyro_signals,
    "gyro",
    TARGET_RATE_HZ,
)

X_frequency = np.concatenate(
    [
        accel_frequency_features,
        gyro_frequency_features,
    ],
    axis=1,
)

frequency_feature_names = (
    accel_frequency_names
    + gyro_frequency_names
)

X_baseline_frequency = np.concatenate(
    [
        X_baseline,
        X_frequency,
    ],
    axis=1,
)

baseline_frequency_names = (
    baseline_feature_names
    + frequency_feature_names
)

X_all_engineered = np.concatenate(
    [
        X_baseline,
        X_distribution,
        X_temporal,
        X_frequency,
    ],
    axis=1,
)

all_engineered_names = (
    baseline_feature_names
    + distribution_feature_names
    + temporal_feature_names
    + frequency_feature_names
)

assert X_frequency.shape == (
    len(train_metadata),
    72,
)
assert X_baseline_frequency.shape == (
    len(train_metadata),
    128,
)
assert X_all_engineered.shape == (
    len(train_metadata),
    222,
)
assert np.isfinite(
    X_all_engineered
).all()

feature_sets = {
    "baseline": {
        "label": "Baseline",
        "matrix": X_baseline,
        "names": baseline_feature_names,
    },
    "distribution": {
        "label": (
            "Baseline + distribution/correlation"
        ),
        "matrix": X_baseline_distribution,
        "names": baseline_distribution_names,
    },
    "temporal": {
        "label": (
            "Baseline + temporal dynamics"
        ),
        "matrix": X_baseline_temporal,
        "names": baseline_temporal_names,
    },
    "frequency": {
        "label": (
            "Baseline + frequency structure"
        ),
        "matrix": X_baseline_frequency,
        "names": baseline_frequency_names,
    },
    "all_engineered": {
        "label": (
            "All engineered groups"
        ),
        "matrix": X_all_engineered,
        "names": all_engineered_names,
    },
}

candidate_overview = pd.DataFrame([
    {
        "candidate": config["label"],
        "examples": config[
            "matrix"
        ].shape[0],
        "features": config[
            "matrix"
        ].shape[1],
        "finite": bool(
            np.isfinite(
                config["matrix"]
            ).all()
        ),
    }
    for config in feature_sets.values()
])

display(candidate_overview)

print(
    "All predefined feature matrices constructed."
)

,candidate,examples,features,finite
0,Baseline,25585,56,True
1,Baseline + distribution/correlation,25585,110,True
2,Baseline + temporal dynamics,25585,96,True
3,Baseline + frequency structure,25585,128,True
4,All engineered groups,25585,222,True


All predefined feature matrices constructed.


## 5. Compare engineered-feature candidates

Each candidate is evaluated with the same five subject folds and unchanged
Random Forest configuration. The baseline is rerun in this notebook so every
candidate has a directly comparable fold-level reference.

The experiment evaluates only the five feature sets defined before model
results were observed.

In [9]:
from time import perf_counter

from sklearn.ensemble import (
    RandomForestClassifier,
)
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    recall_score,
)


ACTIVITY_CODES = preprocessing_meta[
    "labels"
]["activity_codes_in_order"]

REQUIRED_MACRO_F1_GAIN = 0.01
REQUIRED_IMPROVED_FOLDS = 3


def calculate_metrics(y_true, y_pred):
    return {
        "accuracy": accuracy_score(
            y_true,
            y_pred,
        ),
        "balanced_accuracy": recall_score(
            y_true,
            y_pred,
            labels=ACTIVITY_CODES,
            average="macro",
            zero_division=0,
        ),
        "macro_f1": f1_score(
            y_true,
            y_pred,
            labels=ACTIVITY_CODES,
            average="macro",
            zero_division=0,
        ),
    }


candidate_oof_predictions = {}
fold_metric_rows = []
run_rows = []

for candidate_key, config in feature_sets.items():
    X_candidate = config["matrix"]

    candidate_predictions = np.empty_like(y)
    prediction_filled = np.zeros(
        len(y),
        dtype=bool,
    )

    print(f"\nEvaluating {config['label']}")

    for fold in sorted(
        np.unique(fold_assignments)
    ):
        training_mask = (
            fold_assignments != fold
        )
        validation_mask = (
            fold_assignments == fold
        )

        training_subjects = set(
            subject_ids[training_mask]
        )
        validation_subjects = set(
            subject_ids[validation_mask]
        )

        assert training_subjects.isdisjoint(
            validation_subjects
        )

        fold_model = RandomForestClassifier(
            n_estimators=300,
            random_state=42,
            n_jobs=-1,
        )

        start_time = perf_counter()

        fold_model.fit(
            X_candidate[training_mask],
            y[training_mask],
        )

        fold_predictions = fold_model.predict(
            X_candidate[validation_mask]
        )

        elapsed_seconds = (
            perf_counter() - start_time
        )

        candidate_predictions[
            validation_mask
        ] = fold_predictions

        prediction_filled[
            validation_mask
        ] = True

        fold_metric_rows.append({
            "candidate_key": candidate_key,
            "candidate": config["label"],
            "fold": int(fold),
            **calculate_metrics(
                y[validation_mask],
                fold_predictions,
            ),
        })

        run_rows.append({
            "candidate_key": candidate_key,
            "candidate": config["label"],
            "fold": int(fold),
            "features": X_candidate.shape[1],
            "training_examples": int(
                training_mask.sum()
            ),
            "validation_examples": int(
                validation_mask.sum()
            ),
            "fit_predict_seconds": (
                elapsed_seconds
            ),
        })

        print(
            f"  Fold {fold}: "
            f"{int(validation_mask.sum()):,} "
            f"predictions in "
            f"{elapsed_seconds:.2f} seconds"
        )

    assert prediction_filled.all()

    candidate_prediction_table = (
        train_metadata[
            [
                "array_index",
                "pair_id",
                "user_id",
                "activity_code",
                "fold",
            ]
        ]
        .rename(
            columns={
                "activity_code": "y_true",
            }
        )
        .copy()
    )

    candidate_prediction_table[
        "y_pred"
    ] = candidate_predictions

    candidate_oof_predictions[
        candidate_key
    ] = candidate_prediction_table


candidate_fold_metrics = pd.DataFrame(
    fold_metric_rows
)

candidate_run_log = pd.DataFrame(
    run_rows
)

print("\nAll candidate evaluations completed.")


Evaluating Baseline
  Fold 1: 5,670 predictions in 2.09 seconds
  Fold 2: 4,970 predictions in 2.10 seconds
  Fold 3: 4,970 predictions in 2.15 seconds
  Fold 4: 4,970 predictions in 2.12 seconds
  Fold 5: 5,005 predictions in 2.10 seconds

Evaluating Baseline + distribution/correlation
  Fold 1: 5,670 predictions in 2.82 seconds
  Fold 2: 4,970 predictions in 3.02 seconds
  Fold 3: 4,970 predictions in 2.98 seconds
  Fold 4: 4,970 predictions in 2.96 seconds
  Fold 5: 5,005 predictions in 2.91 seconds

Evaluating Baseline + temporal dynamics
  Fold 1: 5,670 predictions in 2.47 seconds
  Fold 2: 4,970 predictions in 2.63 seconds
  Fold 3: 4,970 predictions in 2.57 seconds
  Fold 4: 4,970 predictions in 2.59 seconds
  Fold 5: 5,005 predictions in 2.55 seconds

Evaluating Baseline + frequency structure
  Fold 1: 5,670 predictions in 2.89 seconds
  Fold 2: 4,970 predictions in 3.44 seconds
  Fold 3: 4,970 predictions in 3.09 seconds
  Fold 4: 4,970 predictions in 3.03 seconds
  Fold 5: 5

In [10]:
CANDIDATE_ORDER = list(
    feature_sets
)

fold_macro_f1_comparison = (
    candidate_fold_metrics
    .pivot(
        index="fold",
        columns="candidate_key",
        values="macro_f1",
    )
    .reindex(
        columns=CANDIDATE_ORDER
    )
)

baseline_fold_macro_f1 = (
    fold_macro_f1_comparison[
        "baseline"
    ]
)

summary_rows = []

for candidate_key in CANDIDATE_ORDER:
    config = feature_sets[
        candidate_key
    ]

    candidate_metrics = (
        candidate_fold_metrics.loc[
            candidate_fold_metrics[
                "candidate_key"
            ].eq(candidate_key)
        ]
    )

    metric_summary = (
        candidate_metrics[
            [
                "accuracy",
                "balanced_accuracy",
                "macro_f1",
            ]
        ]
        .agg(["mean", "std"])
    )

    macro_f1_gain = (
        metric_summary.loc[
            "mean",
            "macro_f1",
        ]
        - baseline_fold_macro_f1.mean()
    )

    folds_improved = int(
        (
            fold_macro_f1_comparison[
                candidate_key
            ]
            > baseline_fold_macro_f1
        ).sum()
    )

    passes_complexity_gate = (
        candidate_key != "baseline"
        and macro_f1_gain
        >= REQUIRED_MACRO_F1_GAIN
        and folds_improved
        >= REQUIRED_IMPROVED_FOLDS
    )

    summary_rows.append({
        "candidate_key": candidate_key,
        "candidate": config["label"],
        "features": config[
            "matrix"
        ].shape[1],
        "accuracy_mean": (
            metric_summary.loc[
                "mean",
                "accuracy",
            ]
        ),
        "accuracy_std": (
            metric_summary.loc[
                "std",
                "accuracy",
            ]
        ),
        "balanced_accuracy_mean": (
            metric_summary.loc[
                "mean",
                "balanced_accuracy",
            ]
        ),
        "balanced_accuracy_std": (
            metric_summary.loc[
                "std",
                "balanced_accuracy",
            ]
        ),
        "macro_f1_mean": (
            metric_summary.loc[
                "mean",
                "macro_f1",
            ]
        ),
        "macro_f1_std": (
            metric_summary.loc[
                "std",
                "macro_f1",
            ]
        ),
        "macro_f1_gain": (
            macro_f1_gain
        ),
        "folds_improved": (
            folds_improved
        ),
        "passes_complexity_gate": (
            passes_complexity_gate
        ),
    })

candidate_summary = pd.DataFrame(
    summary_rows
)

display(
    fold_macro_f1_comparison.round(4)
)

display(
    candidate_summary[
        [
            "candidate",
            "features",
            "accuracy_mean",
            "balanced_accuracy_mean",
            "macro_f1_mean",
            "macro_f1_std",
            "macro_f1_gain",
            "folds_improved",
            "passes_complexity_gate",
        ]
    ].round(4)
)

passing_candidates = (
    candidate_summary.loc[
        candidate_summary[
            "passes_complexity_gate"
        ]
    ]
    .sort_values(
        "macro_f1_mean",
        ascending=False,
    )
)

if passing_candidates.empty:
    print(
        "No engineered candidate passed "
        "the predefined complexity gate."
    )
else:
    best_passing_candidate = (
        passing_candidates.iloc[0]
    )

    print(
        "Best passing candidate:",
        best_passing_candidate[
            "candidate"
        ],
    )

candidate_key,baseline,distribution,temporal,frequency,all_engineered
fold,,,,,
1,0.7382,0.7554,0.7516,0.7622,0.7673
2,0.6229,0.6409,0.6394,0.6564,0.6634
3,0.7372,0.7512,0.7520,0.7643,0.7675
4,0.6537,0.6695,0.6791,0.6897,0.6992
5,0.6892,0.7052,0.7130,0.7360,0.7310


,candidate,features,accuracy_mean,balanced_accuracy_mean,macro_f1_mean,macro_f1_std,macro_f1_gain,folds_improved,passes_complexity_gate
0,Baseline,56,0.6899,0.6909,0.6882,0.0509,0.0000,0,False
1,Baseline + distribution/correlation,110,0.7047,0.7057,0.7044,0.0501,0.0162,5,True
2,Baseline + temporal dynamics,96,0.7083,0.7093,0.7070,0.0485,0.0188,5,True
3,Baseline + frequency structure,128,0.7223,0.7231,0.7217,0.0473,0.0335,5,True
4,All engineered groups,222,0.7261,0.7270,0.7257,0.0449,0.0374,5,True


Best passing candidate: All engineered groups


## 6. Focused diagnostics for the strongest candidates

Frequency features account for most of the improvement, while the complete
engineered set achieves the highest mean macro-F1.

To decide whether the additional distribution and temporal features justify
their complexity, we compare only three candidates:

- the 56-feature baseline;
- baseline plus frequency structure;
- all 222 engineered features.

We inspect per-activity F1 and held-out-subject macro-F1. No further model
variants are introduced.

In [11]:
from sklearn.metrics import (
    precision_recall_fscore_support,
)


ACTIVITY_MAP = {
    "A": "Walking",
    "B": "Jogging",
    "C": "Stairs",
    "D": "Sitting",
    "E": "Standing",
    "F": "Typing",
    "G": "Brushing Teeth",
    "H": "Eating Soup",
    "I": "Eating Chips",
    "J": "Eating Pasta",
    "K": "Drinking from Cup",
    "L": "Eating Sandwich",
    "M": "Kicking Soccer Ball",
    "O": "Playing Catch",
    "P": "Dribbling Basketball",
    "Q": "Writing",
    "R": "Clapping",
    "S": "Folding Clothes",
}

DIAGNOSTIC_CANDIDATES = [
    "baseline",
    "frequency",
    "all_engineered",
]

activity_rows = []

for candidate_key in DIAGNOSTIC_CANDIDATES:
    predictions = (
        candidate_oof_predictions[
            candidate_key
        ]
    )

    (
        precision,
        recall,
        f1,
        support,
    ) = precision_recall_fscore_support(
        predictions["y_true"],
        predictions["y_pred"],
        labels=ACTIVITY_CODES,
        zero_division=0,
    )

    for activity_index, activity_code in enumerate(
        ACTIVITY_CODES
    ):
        activity_rows.append({
            "candidate_key": candidate_key,
            "activity_code": activity_code,
            "activity_name": ACTIVITY_MAP[
                activity_code
            ],
            "f1": f1[activity_index],
            "support": int(
                support[activity_index]
            ),
        })

activity_metrics = pd.DataFrame(
    activity_rows
)

activity_f1_comparison = (
    activity_metrics
    .pivot(
        index=[
            "activity_code",
            "activity_name",
        ],
        columns="candidate_key",
        values="f1",
    )
    .reset_index()
)

activity_f1_comparison[
    "frequency_gain"
] = (
    activity_f1_comparison["frequency"]
    - activity_f1_comparison["baseline"]
)

activity_f1_comparison[
    "all_engineered_gain"
] = (
    activity_f1_comparison[
        "all_engineered"
    ]
    - activity_f1_comparison["baseline"]
)

activity_f1_comparison[
    "all_vs_frequency"
] = (
    activity_f1_comparison[
        "all_engineered"
    ]
    - activity_f1_comparison["frequency"]
)

display(
    activity_f1_comparison
    .sort_values(
        "all_engineered_gain",
        ascending=False,
    )
    .round(3)
)

candidate_key,activity_code,activity_name,all_engineered,baseline,frequency,frequency_gain,all_engineered_gain,all_vs_frequency
6,G,Brushing Teeth,0.889,0.780,0.878,0.097,0.109,0.011
0,A,Walking,0.783,0.721,0.781,0.060,0.062,0.002
16,R,Clapping,0.875,0.820,0.869,0.049,0.055,0.007
10,K,Drinking from Cup,0.606,0.552,0.600,0.048,0.053,0.006
2,C,Stairs,0.725,0.673,0.717,0.044,0.052,0.008
12,M,Kicking Soccer Ball,0.833,0.784,0.807,0.023,0.049,0.026
5,F,Typing,0.755,0.713,0.751,0.038,0.042,0.004
13,O,Playing Catch,0.876,0.838,0.876,0.038,0.038,0.000
11,L,Eating Sandwich,0.287,0.252,0.276,0.025,0.035,0.011
7,H,Eating Soup,0.630,0.595,0.619,0.024,0.035,0.011


In [12]:
subject_rows = []

for candidate_key in DIAGNOSTIC_CANDIDATES:
    predictions = (
        candidate_oof_predictions[
            candidate_key
        ]
    )

    for user_id, subject_predictions in (
        predictions.groupby(
            "user_id",
            sort=True,
        )
    ):
        labels_present = [
            activity_code
            for activity_code in ACTIVITY_CODES
            if activity_code
            in set(
                subject_predictions[
                    "y_true"
                ]
            )
        ]

        subject_rows.append({
            "candidate_key": candidate_key,
            "user_id": int(user_id),
            "macro_f1": f1_score(
                subject_predictions[
                    "y_true"
                ],
                subject_predictions[
                    "y_pred"
                ],
                labels=labels_present,
                average="macro",
                zero_division=0,
            ),
        })

subject_metrics = pd.DataFrame(
    subject_rows
)

subject_f1_comparison = (
    subject_metrics
    .pivot(
        index="user_id",
        columns="candidate_key",
        values="macro_f1",
    )
)

subject_f1_comparison[
    "frequency_gain"
] = (
    subject_f1_comparison["frequency"]
    - subject_f1_comparison["baseline"]
)

subject_f1_comparison[
    "all_engineered_gain"
] = (
    subject_f1_comparison[
        "all_engineered"
    ]
    - subject_f1_comparison["baseline"]
)

subject_f1_comparison[
    "all_vs_frequency"
] = (
    subject_f1_comparison[
        "all_engineered"
    ]
    - subject_f1_comparison["frequency"]
)


def summarize_subject_gain(
    column,
    comparison,
):
    gain = comparison[column]

    return {
        "comparison": column,
        "mean_gain": gain.mean(),
        "median_gain": gain.median(),
        "minimum_gain": gain.min(),
        "maximum_gain": gain.max(),
        "subjects_improved": int(
            gain.gt(0).sum()
        ),
        "subjects_worsened": int(
            gain.lt(0).sum()
        ),
    }


subject_gain_summary = pd.DataFrame([
    summarize_subject_gain(
        "frequency_gain",
        subject_f1_comparison,
    ),
    summarize_subject_gain(
        "all_engineered_gain",
        subject_f1_comparison,
    ),
    summarize_subject_gain(
        "all_vs_frequency",
        subject_f1_comparison,
    ),
])

display(
    subject_gain_summary.round(4)
)

,comparison,mean_gain,median_gain,minimum_gain,maximum_gain,subjects_improved,subjects_worsened
0,frequency_gain,0.0375,0.0303,-0.0213,0.1246,35,6
1,all_engineered_gain,0.0408,0.0347,-0.0390,0.1489,38,3
2,all_vs_frequency,0.0033,0.0065,-0.0625,0.0348,25,16


## 7. Engineered-feature findings

Every predefined engineered feature group passes the practical complexity gate.

| Feature set | Features | Mean macro-F1 | Fold std | Gain |
|---|---:|---:|---:|---:|
| Baseline | 56 | 0.6882 | 0.0509 | — |
| Distribution/correlation | 110 | 0.7044 | 0.0501 | +0.0162 |
| Temporal dynamics | 96 | 0.7070 | 0.0485 | +0.0188 |
| Frequency structure | 128 | 0.7217 | 0.0473 | +0.0335 |
| All engineered groups | 222 | **0.7257** | **0.0449** | **+0.0374** |

Frequency-domain features provide most of the improvement, confirming that
periodicity, spectral concentration, and frequency-band allocation contain
useful information beyond whole-window distribution summaries.

The full engineered representation performs best overall. Compared with the
baseline, it improves all five folds, 17 of 18 activities, and 38 of 41
held-out development subjects. Mean subject-level macro-F1 improves by 0.0408
and the median improvement is 0.0347.

Brushing Teeth receives the largest activity gain, followed by Walking,
Clapping, Drinking from Cup, and Stairs. Writing is the only activity that
declines, by approximately 0.007 F1. Eating Sandwich improves from 0.252 to
0.287 but remains the weakest activity.

The complete representation improves mean macro-F1 by approximately 0.004 over
the frequency-only candidate. This additional gain is smaller than the main
frequency contribution, but it occurs in four of five folds and for 25 of 41
subjects. The extra computational cost is modest for this dataset.

The 222-feature Random Forest is therefore selected as the new classical
development benchmark:

**mean subject-fold macro-F1: 0.7257 ± 0.0449**

This selection applies only to development-model comparison. No test data were
loaded or evaluated.

In [13]:
MODEL_ARTIFACT_DIR = (
    PROJECT_ROOT
    / "modeling"
    / "artifacts"
)

MODEL_ARTIFACT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

ENGINEERED_FEATURE_PATH = (
    MODEL_ARTIFACT_DIR
    / "engineered_222_train.parquet"
)

ENGINEERED_META_PATH = (
    MODEL_ARTIFACT_DIR
    / "engineered_222_meta.json"
)

assert X_all_engineered.shape == (
    len(train_metadata),
    222,
)
assert len(all_engineered_names) == 222
assert len(set(all_engineered_names)) == 222
assert np.isfinite(
    X_all_engineered
).all()
assert np.array_equal(
    train_metadata[
        "array_index"
    ].to_numpy(),
    np.arange(len(train_metadata)),
)

engineered_feature_table = (
    pd.DataFrame(
        X_all_engineered,
        columns=all_engineered_names,
    )
)

engineered_feature_table.insert(
    0,
    "array_index",
    train_metadata[
        "array_index"
    ].to_numpy(),
)

engineered_feature_table.to_parquet(
    ENGINEERED_FEATURE_PATH,
    index=False,
)

engineered_feature_meta = {
    "artifact": str(
        ENGINEERED_FEATURE_PATH.relative_to(
            PROJECT_ROOT
        )
    ),
    "source_sequences": (
        "modified/watch_pair_20hz/"
        "train_sequences.npz"
    ),
    "source_metadata": (
        "modified/watch_pair_20hz/"
        "train_metadata.parquet"
    ),
    "row_key": "array_index",
    "rows": int(
        X_all_engineered.shape[0]
    ),
    "features": int(
        X_all_engineered.shape[1]
    ),
    "dtype": str(
        X_all_engineered.dtype
    ),
    "feature_groups": {
        "baseline": 56,
        "distribution_correlation": 54,
        "temporal_dynamics": 40,
        "frequency_structure": 72,
    },
    "feature_names": list(
        all_engineered_names
    ),
}

ENGINEERED_META_PATH.write_text(
    json.dumps(
        engineered_feature_meta,
        indent=2,
    ),
    encoding="utf-8",
)

print(
    "Saved feature table:",
    ENGINEERED_FEATURE_PATH,
)
print(
    "Shape:",
    engineered_feature_table.shape,
)
print(
    "Saved metadata:",
    ENGINEERED_META_PATH,
)

Saved feature table: /Users/yairlutaty/dev/yair.lutaty/projects/activity_project/modeling/artifacts/engineered_222_train.parquet
Shape: (25585, 223)
Saved metadata: /Users/yairlutaty/dev/yair.lutaty/projects/activity_project/modeling/artifacts/engineered_222_meta.json
